# Métodos Numéricos — Solução Numérica de EDOs

## O Método de Euler

> **Como usar no Colab:** abra este caderno no Google Colab, execute as células
> de cima para baixo (`Shift + Enter`) e preencha os trechos marcados com
> `# TODO`. Você pode usar o menu *Ambiente de execução → Reiniciar e executar
> tudo* sempre que quiser começar do zero.

Este caderno já vem com:

* o **enunciado** do exercício;
* uma **explicação básica** da ideia do método;
* um **esqueleto de código** para você completar (a função `euleredo`);
* o **código de desenho pronto** (você não precisa estudar bibliotecas
  gráficas — o foco é implementar Euler e comparar com o resultado do Python);
* **três EDOs sugeridas** para testar a sua implementação;
* um **resumo de outro método** (Heun / Runge–Kutta) com um pequeno
  exercício adicional.


## 1. Enunciado

Sua tarefa será implementar uma função `euleredo` que recebe:

* uma equação diferencial `d` (a *derivada* $y' = d(x,y)$, que pode ser
  implementada como uma função anônima nas variáveis `x` e `y`);
* um vetor `x` de pontos (os pontos onde queremos estimar os valores `y` da
  função);
* a condição inicial `y0`, ou seja, o valor da função que estamos estimando
  no ponto `x[0]`.

Sua função devolve um vetor `y` com os valores numéricos da função aplicada
nos pontos `x`, partindo da condição inicial `y0`.

A função aplica o **método de Euler**, ou seja:

* define explicitamente o valor de `y[0]` para `x[0]`, usando a condição
  inicial;
* executa um laço `for` para preencher os demais valores do vetor `y`. Em cada
  iteração um novo valor é calculado a partir do valor anterior, da inclinação
  da função (dada pelo parâmetro `d`) e do **intervalinho** $h$ entre pontos
  adjacentes de `x` (você deverá calcular esse $h$ dentro da sua função).

Você deverá testar a sua função `euleredo` para as EDOs de exemplo deste caderno.
Faça **ao menos três testes**, com quantidades distintas de pontos no
`linspace` de `x`, e observe como o erro diminui com intervalinhos menores.
Escreva um **relatório sucinto** (aqui mesmo, no caderno) mostrando os
desenhos da função original e dos pontos calculados pela sua função.

Para cada teste, diga claramente qual é o valor inicial **`y0`**.


## 2. A ideia do método de Euler (explicação básica)

Queremos resolver um **problema de valor inicial**:

$$\frac{dy}{dx} = d(x,y), \qquad y(x_0) = y_0 .$$

Conhecemos a *inclinação* da solução em qualquer ponto — é exatamente o que a
função `d(x, y)` nos dá — mas não conhecemos a função $y(x)$ em si.

A intuição do método de Euler é simples: **se eu conheço um ponto da curva e
a inclinação ali, dou um pequeno passo nessa direção (uma reta tangente) e
chego, aproximadamente, a um novo ponto da curva.** Repetindo o passo
sucessivas vezes, "caminho" sobre uma poligonal que acompanha a solução.

Partindo de $(x_i, y_i)$ e usando um passo $h = x_{i+1} - x_i$:

$$\boxed{\,y_{i+1} = y_i + h \cdot d(x_i,\, y_i)\,}$$

Quanto **menor** o passo $h$, mais a poligonal "gruda" na curva verdadeira —
mas mais passos (e mais contas) são necessários. Investigar esse compromisso é
parte do exercício.


## 3. Preparação do ambiente

Importamos `numpy` (vetores), `matplotlib` (desenho) e `sympy`
(matemática simbólica — usado **apenas** para obter a solução exata de
referência; veja a justificativa na próxima seção).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Deixa os gráficos um pouco maiores e mais legíveis
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## 4. Uma EDO de exemplo e a sua solução exata

Vamos trabalhar com o seguinte problema de valor inicial:

$$\frac{dy}{dx} = x - y, \qquad y(0) = 3 .$$

Precisamos de uma curva "verdadeira" para comparar com a aproximação de Euler.

### Por que obter a solução com `sympy.dsolve`?

Optamos por
[`sympy.dsolve`](https://docs.sympy.org/latest/modules/solvers/ode.html)
pelos seguintes motivos:

1. **É a solução *exata*, não outra aproximação.** `dsolve` resolve a EDO de
   forma *simbólica* e devolve a fórmula fechada $y(x)$. Assim, a "curva
   original" do gráfico é a solução verdadeira, e não mais um método numérico
   competindo com o seu — a comparação fica honesta.
2. **Resolve o problema de valor inicial diretamente**, via o argumento
   `ics` (*initial conditions*), já embutindo `y(0) = 3`.
3. **É um bom momento didático:** soluções fechadas *nem sempre existem* ou
   *nem sempre são fáceis*. Quando `dsolve` não consegue (ou a fórmula é
   inviável), o caminho natural é justamente um método numérico — que é o
   tema do curso. Mais adiante mostramos a alternativa puramente numérica
   `scipy.integrate.solve_ivp`.

Convertemos a expressão simbólica em uma função NumPy com
`sympy.lambdify`, para poder avaliá-la em vetores de pontos.

In [ ]:
# --- EDO de exemplo: y' = x - y, com y(0) = 3 -----------------------------

# A derivada como função anônima (é isto que será passado para `euleredo`):
d_exemplo = lambda x, y: x - y

x0_exemplo = 0.0   # ponto inicial
y0_exemplo = 3.0   # condição inicial (y0 = 3)


def solucao_de_referencia(d_simbolica, x0, y0):
    """Resolve y' = d(x, y) com y(x0) = y0 usando sympy.dsolve e
    devolve uma função NumPy f(x) com a solução EXATA.

    `d_simbolica` é uma função que recebe os símbolos (x, y) do sympy
    e devolve a expressão simbólica da derivada.
    """
    x = sp.symbols("x")
    y = sp.Function("y")
    edo = sp.Eq(y(x).diff(x), d_simbolica(x, y(x)))
    sol = sp.dsolve(edo, y(x), ics={y(sp.sympify(x0)): sp.sympify(y0)})

    # dsolve pode devolver uma lista quando há múltiplos ramos de solução
    if isinstance(sol, list):
        sol = sol[0]

    rhs = sol.rhs
    constantes_livres = rhs.free_symbols - {x}
    if constantes_livres:
        raise ValueError(
            f"dsolve não conseguiu aplicar a condição inicial: "
            f"constantes livres em rhs: {constantes_livres}"
        )

    print("Solução exata encontrada por sympy.dsolve:")
    sp.pprint(sp.Eq(y(x), sp.simplify(rhs)))
    return sp.lambdify(x, rhs, "numpy")


# Para o exemplo, a derivada simbólica tem a mesma "cara" da anônima:
f_exata = solucao_de_referencia(lambda x, y: x - y, x0_exemplo, y0_exemplo)

# Conferindo: f_exata(0) deve valer 3
print("\nf_exata(0) =", f_exata(0.0), "(esperado: 3.0)")


## 5. Esqueleto da função `euleredo` — **para você completar**

Complete os trechos marcados com `# TODO`. Não altere a assinatura da função.

Dicas:

* `y = np.zeros_like(x, dtype=float)` cria um vetor de zeros do mesmo tamanho
  de `x` (bom lugar para guardar os resultados).
* O primeiro valor, `y[0]`, vem **diretamente** da condição inicial `y0`.
* O passo entre os pontos `i` e `i+1` é `h = x[i+1] - x[i]`. Calcular o passo
  *dentro* do laço permite que sua função funcione mesmo se os pontos não
  forem perfeitamente uniformes.
* A fórmula de atualização é $y_{i+1} = y_i + h \cdot d(x_i, y_i)$.

In [ ]:
def euleredo(d, x, y0):
    """Resolve y' = d(x, y) pelo método de Euler.

    Parâmetros
    ----------
    d  : callable
        Função que recebe dois números (x_i, y_i) e devolve a inclinação
        da solução naquele ponto, ou seja, dy/dx = d(x, y).
        Exemplo: lambda x, y: x - y
    x  : sequência de números (lista ou np.ndarray)
        Pontos do eixo x onde queremos calcular a solução.
        Use np.linspace(a, b, n) para gerar n pontos igualmente espaçados
        entre a e b.  O primeiro elemento, x[0], é o ponto onde a condição
        inicial y0 é aplicada.
    y0 : float
        Valor da solução no ponto inicial, isto é, y(x[0]) = y0.

    Devolve
    -------
    y : np.ndarray de mesmo tamanho que x
        y[i] é a estimativa de Euler para a solução no ponto x[i].
        y[0] == y0 por definição; os demais são calculados pelo laço.

    Como funciona internamente
    --------------------------
    * `np.asarray(x, dtype=float)` garante que x seja um array NumPy de
      números reais, independentemente de se ter passado uma lista
      Python ou um array já pronto.
    * `np.zeros_like(x, dtype=float)` cria um vetor de zeros com o mesmo
      número de elementos que x — é nesse vetor que os resultados serão
      armazenados posição a posição.
    * O laço avança de i=0 até i=len(x)-2, calculando y[i+1] a partir de
      y[i] a cada passo.
    """
    # Converte x para array NumPy de float (aceita lista, tupla, array, etc.)
    x = np.asarray(x, dtype=float)

    # Cria o vetor de resultados, inicialmente todo zeros, com len(x) posições
    y = np.zeros_like(x, dtype=float)

    # 1) Defina explicitamente o valor inicial:
    # TODO: y[0] = ...

    # 2) Laço for para preencher os demais valores:
    #    i percorre 0, 1, 2, ..., len(x)-2
    #    a cada iteração calculamos y[i+1] a partir de y[i]
    for i in range(len(x) - 1):
        h = None       # TODO: calcule o intervalinho entre x[i] e x[i+1]
        # TODO: y[i + 1] = ...   (aplique a fórmula de Euler)
        pass

    return y


## 6. Código de desenho — **já fornecido**

Você **não precisa** mexer aqui. Esta função desenha:

* a **curva exata** (linha contínua), avaliada numa malha bem fina;
* os **pontos calculados pela sua `euleredo`** (marcadores ligados por
  segmentos — a "poligonal de Euler").

Ela também devolve o **erro máximo absoluto** entre a sua aproximação e a
solução exata nos pontos de `x`, para você acompanhar numericamente a
melhora quando $h$ diminui.

In [ ]:
def plota_comparacao(d, f_exata, x, y0, titulo=""):
    """Desenha a solução exata e a aproximação de Euler, e devolve o erro
    máximo absoluto nos pontos de x."""
    y_aprox = euleredo(d, x, y0)

    # malha fina só para desenhar a curva exata "lisa"
    x_fino = np.linspace(x[0], x[-1], 400)
    y_fino = f_exata(x_fino)

    plt.figure()
    plt.plot(x_fino, y_fino, "-", linewidth=2, label="Solução exata (sympy)")
    plt.plot(x, y_aprox, "o--", markersize=5,
             label=f"Euler ({len(x)} pontos)")
    plt.xlabel("x"); plt.ylabel("y")
    plt.title(titulo if titulo else "Euler vs. solução exata")
    plt.legend()
    plt.show()

    erro_max = np.max(np.abs(y_aprox - f_exata(x)))
    h = x[1] - x[0]
    print(f"n = {len(x):4d} pontos | h = {h:.5f} | "
          f"erro máximo absoluto = {erro_max:.6e}")
    return erro_max


## 7. Relatório sucinto — **preencha aqui**

> **Como montar o relatório:** antes de responder, execute as células da
> **Seção 8** (as três EDOs sugeridas) variando o número de pontos e
> observando os gráficos e erros impressos. Use esses resultados como base
> para as suas respostas abaixo.

> **Como o erro máximo absoluto é calculado:** a função `plota_comparacao`
> já o calcula e imprime automaticamente ao final de cada execução. O valor
> impresso é:
> $$\max_i \bigl|y_{\text{Euler}}(x_i) - y_{\text{exata}}(x_i)\bigr|,$$
> ou seja, a maior diferença em módulo entre a estimativa de Euler e a
> solução exata, dentre todos os pontos $x_i$ do vetor `x`. Você não precisa
> calculá-lo manualmente — basta ler o número impresso pela função.

Responda em poucas linhas (clique duas vezes nesta célula para editar):

1. **O que acontece com a poligonal de Euler quando aumentamos o número de
   pontos?** _(sua resposta)_

2. **Para cada uma das três EDOs separadamente: como o erro máximo absoluto
   se comporta quando o passo $h$ é reduzido à metade?** Aproximadamente por
   qual fator ele cai em cada caso? O fator é parecido nas três EDOs, ou
   depende da função? _(sua resposta para EDO 1 / EDO 2 / EDO 3)_

3. **Das três EDOs testadas, em qual delas o método de Euler apresentou o
   maior erro para um mesmo número de pontos?** Por que você acha que essa
   EDO é mais difícil de aproximar do que as outras? _(sua resposta)_


## 8. Três EDOs sugeridas para testar a sua implementação

Use o **mesmo** `y0 = 3` em todas. Cada uma ilustra um fenômeno diferente —
rode sua `euleredo` em cada caso, varie o número de pontos e comente o que
observa.

| # | EDO | Solução exata | Intervalo sugerido | O que ela ilustra |
|---|-----|---------------|--------------------|-------------------|
| 1 | $y' = -2\,y$ | $y(x) = 3\,e^{-2x}$ | $[0,\,3]$ | **Decaimento e estabilidade.** Com passos grandes ($h \gtrsim 1$) a poligonal de Euler pode oscilar e até divergir, embora a solução verdadeira tenda suavemente a zero. |
| 2 | $y' = x\,y$ | $y(x) = 3\,e^{x^{2}/2}$ | $[0,\,2]$ | **Crescimento rápido.** O erro cresce depressa; é preciso $h$ bem pequeno para acompanhar a curva. |
| 3 | $y' = 0{,}5\,y\left(1 - \dfrac{y}{8}\right)$ | $y(x) = \dfrac{8}{1 + \tfrac{5}{3}\,e^{-0{,}5x}}$ | $[0,\,15]$ | **Equação logística (não linear).** A solução satura na capacidade de suporte $K = 8$; sabor de modelagem populacional/ambiental. |

As três soluções exatas acima podem ser obtidas com a mesma
`solucao_de_referencia(...)` (que usa `sympy.dsolve`). A célula seguinte já
deixa tudo pronto: basta escolher o caso na variável `caso`.

In [ ]:
# Catálogo das três EDOs sugeridas: (derivada anônima, derivada simbólica,
#                                       x0, y0, intervalo [a, b], rótulo)
edos_sugeridas = {
    1: dict(
        d        = lambda x, y: -2.0 * y,
        d_sym    = lambda x, y: -2 * y,
        x0=0.0, y0=3.0, a=0.0, b=3.0,
        rotulo="EDO 1:  y' = -2y,  y(0)=3   (decaimento)",
    ),
    2: dict(
        d        = lambda x, y: x * y,
        d_sym    = lambda x, y: x * y,
        x0=0.0, y0=3.0, a=0.0, b=2.0,
        rotulo="EDO 2:  y' = x·y,  y(0)=3   (crescimento)",
    ),
    3: dict(
        d        = lambda x, y: 0.5 * y * (1 - y / 8.0),
        d_sym    = lambda x, y: sp.Rational(1, 2) * y * (1 - y / 8),
        x0=0.0, y0=3.0, a=0.0, b=15.0,
        rotulo="EDO 3:  logística,  y(0)=3   (saturação em K=8)",
    ),
}

# >>> Escolha o caso que quer testar: 1, 2 ou 3 <<<
caso = 1
cfg = edos_sugeridas[caso]

f_exata_caso = solucao_de_referencia(cfg["d_sym"], cfg["x0"], cfg["y0"])

for n in (6, 21, 81):
    x = np.linspace(cfg["a"], cfg["b"], n)
    plota_comparacao(cfg["d"], f_exata_caso, x, cfg["y0"],
                     titulo=cfg["rotulo"] + f"   —   {n} pontos")


## 9. Resumo — existem outros métodos (Heun / Runge–Kutta)

O método de Euler é o membro mais simples de uma família de **métodos de
passo único**, que avançam de $x_i$ para $x_{i+1}$ usando informação sobre a
inclinação. A diferença entre eles está em **quantas vezes** e **onde** a
derivada $d(x,y)$ é avaliada dentro de cada passo:

* **Euler (ordem 1).** Usa a inclinação apenas no início do passo. Erro global
  proporcional a $h$. Simples, mas exige passos pequenos para boa precisão.

* **Heun / Euler melhorado (ordem 2, um Runge–Kutta de 2 estágios).** Calcula
  a inclinação no início do passo, dá um "passo de teste" de Euler, calcula a
  inclinação no fim e usa a **média** das duas. Erro global proporcional a
  $h^{2}$ — reduzir $h$ pela metade derruba o erro a, grosso modo, um quarto.

* **Runge–Kutta clássico (RK4, ordem 4).** Combina quatro avaliações da
  derivada por passo; é o "cavalo de batalha" tradicional, com erro global
  proporcional a $h^{4}$.

* **Na prática (profissional):**
  [`scipy.integrate.solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)
  implementa, por padrão, um Runge–Kutta adaptativo (RK45 de
  Dormand–Prince), que ajusta o passo automaticamente. É o que se usa em
  trabalho real — mas implementar Euler/Heun "na mão" é o que faz entender
  *por que* esses métodos funcionam.

### Fórmula do método de Heun

Início (preditor, um passo de Euler):

$$\tilde{y}_{i+1} = y_i + h\, d(x_i, y_i)$$

Correção (média das inclinações nas duas pontas):

$$\boxed{\; y_{i+1} = y_i + \dfrac{h}{2}\Big[\, d(x_i, y_i) + d\big(x_{i+1},\, \tilde{y}_{i+1}\big) \,\Big] \;}$$


## 10. Exercício bônus — implemente o método de Heun

Complete a função `heunedo` abaixo (mesma assinatura de `euleredo`) usando a
fórmula do preditor–corretor acima. Em seguida, rode a célula de comparação:
para um **mesmo** número de pontos, Heun deve ficar visivelmente mais perto
da solução exata do que Euler.

In [ ]:
def heunedo(d, x, y0):
    """Resolve y' = d(x, y) pelo método de Heun (Euler melhorado, RK de
    2 estágios). Mesma assinatura de euleredo."""
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x, dtype=float)

    # TODO: y[0] = ...

    for i in range(len(x) - 1):
        h = x[i + 1] - x[i]
        # TODO (preditor):  y_tilde = y[i] + h * d(x[i], y[i])
        # TODO (corretor):  y[i+1]  = y[i] + (h/2) * ( d(x[i], y[i])
        #                                              + d(x[i+1], y_tilde) )
        pass

    return y


In [ ]:
# Comparação Euler  x  Heun  x  solução exata, com o MESMO número de pontos
n = 11
x = np.linspace(0.0, 4.0, n)

y_euler = euleredo(d_exemplo, x, y0_exemplo)
y_heun  = heunedo(d_exemplo, x, y0_exemplo)

x_fino = np.linspace(x[0], x[-1], 400)

plt.figure()
plt.plot(x_fino, f_exata(x_fino), "-",  linewidth=2, label="Solução exata")
plt.plot(x, y_euler, "o--", label=f"Euler ({n} pontos)")
plt.plot(x, y_heun,  "s--", label=f"Heun  ({n} pontos)")
plt.xlabel("x"); plt.ylabel("y")
plt.title("Euler  x  Heun  x  exata  —  mesma quantidade de pontos")
plt.legend()
plt.show()

print(f"Erro máximo Euler: {np.max(np.abs(y_euler - f_exata(x))):.6e}")
print(f"Erro máximo Heun : {np.max(np.abs(y_heun  - f_exata(x))):.6e}")
